### Simulating a single electrode for a single participant
- Simulating central line (Cz to C5) for a single participant

#### Imports

In [1]:
# Imports required for functionality 
import numpy as np
from matplotlib import pyplot 

from neurodsp.utils import create_times
from neurodsp.sim import sim_oscillation, set_random_seed, sim_combined
from neurodsp.plts import plot_time_series, plot_power_spectra
from neurodsp.spectral import compute_spectrum_welch

from specparam import SpectralModel
from specparam.plts import plot_spectra

#### Data Simulationn settings

In [2]:
# Set seed for consistency
set_random_seed(256)

# Simulation settings
n_seconds = 8 # Number of seconds
s_rate = 250 # Sampling rate

# Compute an array of time values for plotting, and check length of data
times = create_times(n_seconds, s_rate)

#### Simulate EEG-like signal for electrode Cz

#### Simulating EEG-like signal for central line

In [ ]:
# Define correlations between Cz and central line electrodes
corr_factors = {
    'Cz': 1.0,   # reference electrode
    'C1': 0.8,   # adjacent
    'C3': 0.6,   # farther
    'C5': 0.4    # more distant
}

# Generate signals for each electrode
electrode_signals = {}

for elec, rho in corr_factors.items():
    if elec == 'Cz':
        electrode_signals[elec] = cz_signal
    else:
        # scale noise to achieve desired correlation
        sigma_source = np.std(cz_signal)
        noise_std = np.sqrt((1 - rho**2) / rho**2) * sigma_source
        noise = noise_std * np.random.randn(len(cz_signal))
        electrode_signals[elec] = rho * cz_signal + noise


# Create figure with 4 horizontal subplots
fig, axes = plt.subplots(1, 4, figsize=(18, 5))  # axes is a list of 4 Axes

# List of electrodes in order
electrodes = ['Cz', 'C1', 'C3', 'C5']


# Compute power spectra and plot Spectral Models
for i, elec in enumerate(electrodes):
    sig = electrode_signals[elec]
    freqs, powers = compute_spectrum_welch(sig, s_rate)

    fm_elec = SpectralModel()
    freq_range = [2, 40]
    
    fm_elec.fit(freqs, powers, freq_range)
    fm_elec.plot(ax=axes[i])
    axes[i].set_title(elec)